# Exercises: Deep Learning

Hands-on PyTorch exercises covering custom components and debugging.

## Exercise 1: Custom Loss Function

Implement **Focal Loss** as a custom PyTorch loss function.

Focal Loss down-weights easy examples so the model focuses on hard ones:

$$FL(p_t) = -\alpha_t (1 - p_t)^\gamma \log(p_t)$$

**Requirements:**
- Subclass `nn.Module`
- Support `alpha` and `gamma` parameters
- Test on a small synthetic batch and compare with `nn.CrossEntropyLoss`

In [ ]:
# YOUR CODE HERE

### Solution

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

np.random.seed(42)
torch.manual_seed(42)


class FocalLoss(nn.Module):
    def __init__(self, alpha=1.0, gamma=2.0, reduction='mean'):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, logits, targets):
        ce_loss = F.cross_entropy(logits, targets, reduction='none')
        pt = torch.exp(-ce_loss)  # probability of correct class
        focal_weight = self.alpha * (1 - pt) ** self.gamma
        loss = focal_weight * ce_loss
        if self.reduction == 'mean':
            return loss.mean()
        elif self.reduction == 'sum':
            return loss.sum()
        return loss


# Test
batch_size, n_classes = 16, 4
logits = torch.randn(batch_size, n_classes)
targets = torch.randint(0, n_classes, (batch_size,))

focal = FocalLoss(alpha=1.0, gamma=2.0)
ce = nn.CrossEntropyLoss()

print(f"Focal Loss:         {focal(logits, targets):.4f}")
print(f"Cross-Entropy Loss: {ce(logits, targets):.4f}")

# When gamma=0, focal loss ≈ cross-entropy
focal_g0 = FocalLoss(alpha=1.0, gamma=0.0)
print(f"Focal (gamma=0):    {focal_g0(logits, targets):.4f}  (should ≈ CE)")


### Explanation

Focal Loss multiplies the standard cross-entropy by $(1-p_t)^\gamma$, which approaches 0 for well-classified examples (high $p_t$) and stays near 1 for hard examples. When $\gamma=0$ it reduces to standard CE.

## Exercise 2: Network Architecture Design

Design a configurable MLP with:
- Variable number of hidden layers and widths
- Choice of activation (ReLU, GELU, Tanh)
- Optional dropout and batch normalisation

**Requirements:**
- Accept a list of layer sizes `[input, h1, h2, ..., output]`
- Train on a synthetic dataset for a few epochs to verify it works

In [ ]:
# YOUR CODE HERE

### Solution

In [ ]:
import torch
import torch.nn as nn
import numpy as np

np.random.seed(42)
torch.manual_seed(42)

ACTIVATIONS = {'relu': nn.ReLU, 'gelu': nn.GELU, 'tanh': nn.Tanh}


class ConfigurableMLP(nn.Module):
    def __init__(self, layer_sizes, activation='relu',
                 dropout=0.0, use_batchnorm=False):
        super().__init__()
        act_cls = ACTIVATIONS[activation]
        layers = []
        for i in range(len(layer_sizes) - 1):
            layers.append(nn.Linear(layer_sizes[i], layer_sizes[i + 1]))
            if i < len(layer_sizes) - 2:  # no act/bn/drop after final layer
                if use_batchnorm:
                    layers.append(nn.BatchNorm1d(layer_sizes[i + 1]))
                layers.append(act_cls())
                if dropout > 0:
                    layers.append(nn.Dropout(dropout))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


# Quick training test
from torch.utils.data import TensorDataset, DataLoader

X = torch.randn(500, 10)
y = (X[:, 0] + X[:, 1] > 0).long()

model = ConfigurableMLP([10, 64, 32, 2], activation='gelu',
                         dropout=0.2, use_batchnorm=True)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

loader = DataLoader(TensorDataset(X, y), batch_size=32, shuffle=True)

for epoch in range(5):
    total_loss = 0
    for xb, yb in loader:
        pred = model(xb)
        loss = loss_fn(pred, yb)
        opt.zero_grad(); loss.backward(); opt.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}: loss = {total_loss / len(loader):.4f}")

# Accuracy
model.eval()
with torch.no_grad():
    acc = (model(X).argmax(1) == y).float().mean()
print(f"\nFinal accuracy: {acc:.4f}")


### Explanation

The `layer_sizes` list drives a loop that builds `Linear → [BN] → Act → [Dropout]` blocks. The final layer has no activation so raw logits feed into `CrossEntropyLoss`. Using `nn.Sequential` keeps the code clean.

## Exercise 3: Learning Rate Experiment

Train the **same architecture** with 5 different learning rates and plot the training loss curves side by side.

**Requirements:**
- LRs: `[1e-1, 1e-2, 1e-3, 1e-4, 1e-5]`
- Same init weights for fair comparison (use `model.load_state_dict`)
- 20 epochs each, record loss per epoch
- One figure with all 5 curves

In [ ]:
# YOUR CODE HERE

### Solution

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import copy

np.random.seed(42)
torch.manual_seed(42)

X = torch.randn(400, 8)
y = (X[:, :3].sum(dim=1) > 0).long()

base_model = nn.Sequential(
    nn.Linear(8, 32), nn.ReLU(),
    nn.Linear(32, 16), nn.ReLU(),
    nn.Linear(16, 2),
)
init_state = copy.deepcopy(base_model.state_dict())

lrs = [1e-1, 1e-2, 1e-3, 1e-4, 1e-5]
epochs = 20
loss_fn = nn.CrossEntropyLoss()
all_curves = {}

for lr in lrs:
    model = nn.Sequential(
        nn.Linear(8, 32), nn.ReLU(),
        nn.Linear(32, 16), nn.ReLU(),
        nn.Linear(16, 2),
    )
    model.load_state_dict(copy.deepcopy(init_state))
    opt = torch.optim.SGD(model.parameters(), lr=lr)
    losses = []
    for ep in range(epochs):
        pred = model(X)
        loss = loss_fn(pred, y)
        opt.zero_grad(); loss.backward(); opt.step()
        losses.append(loss.item())
    all_curves[lr] = losses

plt.figure(figsize=(10, 5))
for lr, losses in all_curves.items():
    plt.plot(losses, label=f'lr={lr}')
plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.title('Learning Rate Comparison')
plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('exercises/ex3_lr_experiment.png', dpi=100)
plt.close()
print("Saved exercises/ex3_lr_experiment.png")
for lr, losses in all_curves.items():
    print(f"  lr={lr:.0e} → final loss = {losses[-1]:.4f}")


### Explanation

Loading identical initial weights isolates the effect of the learning rate. Too large → loss diverges; too small → barely moves. The sweet spot converges quickly without oscillation. This is the intuition behind LR range tests and schedulers.

## Exercise 4: Debug a Broken Training Loop

The code below has **4 bugs**. Find and fix them all.

**Hints:** look at gradient handling, data types, loss function choice, and evaluation mode.

In [ ]:
# BROKEN CODE — find and fix the 4 bugs

import torch
import torch.nn as nn

torch.manual_seed(42)
X = torch.randn(200, 5)
y = (X[:, 0] > 0).float()  # BUG 1: target shape/type issue

model = nn.Sequential(
    nn.Linear(5, 16), nn.ReLU(),
    nn.Linear(16, 2),  # BUG 2: wrong output size for binary
)

loss_fn = nn.MSELoss()  # BUG 3: wrong loss for classification
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(50):
    pred = model(X)
    loss = loss_fn(pred, y)
    loss.backward()  # BUG 4: gradients accumulate without zero_grad
    opt.step()
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}: loss = {loss.item():.4f}")

# Evaluate
acc = (model(X).argmax(1) == y).float().mean()
print(f"Accuracy: {acc:.4f}")


### Solution

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(42)
X = torch.randn(200, 5)
y = (X[:, 0] > 0).long()  # FIX 1: use .long() for class indices

model = nn.Sequential(
    nn.Linear(5, 16), nn.ReLU(),
    nn.Linear(16, 2),  # FIX 2: 2 outputs is correct for CrossEntropyLoss
)

loss_fn = nn.CrossEntropyLoss()  # FIX 3: CE for classification
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(50):
    pred = model(X)
    loss = loss_fn(pred, y)
    opt.zero_grad()  # FIX 4: zero gradients before backward
    loss.backward()
    opt.step()
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}: loss = {loss.item():.4f}")

# Evaluate
model.eval()
with torch.no_grad():
    acc = (model(X).argmax(1) == y).float().mean()
print(f"Accuracy: {acc:.4f}")


### Explanation

**Bug 1:** `CrossEntropyLoss` expects `LongTensor` targets, not float.
**Bug 2:** Actually correct — 2 outputs + CE is the standard pattern.
**Bug 3:** MSE is for regression; CE is for classification.
**Bug 4:** Without `opt.zero_grad()`, gradients accumulate across epochs, causing erratic updates. Also, evaluation should use `model.eval()` and `torch.no_grad()`.